# 🌟 SPA-VAEGAN: Complete Master End-to-End Pipeline
### **Structure-Preserving Adaptive VAE-GAN for 4× Single Image Super-Resolution ($64\times 64 \to 256\times 256$)**
**Author:** Bevan (@Bevan)  
**Academic Context:** Advanced Generative AI Continuous Assessment (CAC)  
**Repository:** [SPA-VAEGAN on GitHub](https://github.com/bevanmathewc/SPA-VAEGAN.git)

---

### **Table of Contents & Phase Navigator**
1. [**Phase 0: Environment, Google Drive & Codebase Setup**](#phase-0)
2. [**Phase 1: Dataset Pipeline, Patch Banks & Sanity Verification**](#phase-1)
3. [**Phase 2: Baseline Conditional VAE Training (30,000 Iterations)**](#phase-2)
4. [**Phase 3: Proposed SPA-VAEGAN Adversarial Training (20,000 Iterations)**](#phase-3)
5. [**Phase 4: Quantitative Benchmarking, Figures & Deployment**](#phase-4)

---

### **Key Architectural Principles of This Master Pipeline**
- **Single Tab Simplicity:** No switching between multiple notebooks. Everything from initial environment check to final benchmark tables and Streamlit export lives in this single document.
- **Persistent Cloud Cache:** Raw DIV2K zips, unzipped datasets, memory-mapped patch banks, and training checkpoints are saved directly to **Google Drive (`/content/drive/MyDrive/SPA-VAEGAN/`)**.
- **Automated Resumption:** If Colab disconnects or deletes your runtime, re-running the setup cells and the active training cell automatically resumes from the exact iteration stored in `last.pt` on Google Drive!


<a name="phase-0"></a>
# 🚀 Phase 0: Environment, Google Drive & Codebase Setup

### **Purpose of Code**
Verifies GPU allocation (NVIDIA T4 or better), mounts Google Drive for continuous storage, and clones or syncs the project repository (`src/`, `models/`, `configs/`).


In [ ]:
# ==============================================================================
# ENVIRONMENT & REPOSITORY SETUP (GOOGLE COLAB & LOCAL)
# ==============================================================================
GITHUB_REPO_URL = "https://github.com/bevanmathewc/SPA-VAEGAN.git"

import os
import sys
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except ImportError:
    print("[i] Running in local/offline environment (not Google Colab).")
    IS_COLAB = False

if IS_COLAB:
    REPO_DIR = '/content/drive/MyDrive/SPA-VAEGAN'
    
    # Check if codebase exists in Drive
    if not os.path.exists(os.path.join(REPO_DIR, 'src')):
        print(f"Repository source code not found in Drive at {REPO_DIR}.")
        
        # Check for uploaded zip archive
        zip_candidates = [
            '/content/SPA-VAEGAN.zip',
            '/content/GENAI.zip',
            '/content/drive/MyDrive/SPA-VAEGAN.zip',
            '/content/drive/MyDrive/GENAI.zip'
        ]
        unzipped = False
        for z in zip_candidates:
            if os.path.exists(z):
                print(f"[✓] Found uploaded zip archive: {z}")
                print(f"Extracting into {REPO_DIR}...")
                import zipfile
                os.makedirs(REPO_DIR, exist_ok=True)
                with zipfile.ZipFile(z, 'r') as zf:
                    zf.extractall(REPO_DIR)
                unzipped = True
                print("[✓] Unzipped successfully!")
                break
                
        # If not unzipped, clone from GitHub
        if not unzipped:
            print(f"[✓] Cloning repository from GitHub: {GITHUB_REPO_URL}...")
            !git clone {GITHUB_REPO_URL} {REPO_DIR}

    # Use %cd magic so BOTH Python and !shell commands execute from the repo root
    if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'src')):
        %cd {REPO_DIR}
    elif os.path.exists('/content/src'):
        %cd /content

    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    os.environ["PYTHONPATH"] = os.getcwd()

print(f"\n[✓] Current Working Directory: {os.getcwd()}")
if os.path.exists("src"):
    print("[✓] Project codebase successfully verified (Found 'src', 'models', 'configs').")
else:
    print("[!] Warning: 'src' folder still not found. Please upload 'SPA-VAEGAN.zip' or clone repo.")

# Hardware inspection
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("[!] Warning: No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU.")


### **Install Core Scientific Dependencies**

In [ ]:
# Install required packages quietly
!pip install -q lpips torchmetrics pytorch-fid pyyaml tqdm pandas matplotlib
print("[✓] Dependencies installed successfully.")


<a name="phase-1"></a>
# 📦 Phase 1: Dataset Pipeline, Patch Banks & Sanity Verification

### **Purpose of Code**
1. Downloads the official **DIV2K** dataset to Google Drive:
   - `DIV2K_train_HR.zip` (800 images, ~3.5 GB)
   - `DIV2K_valid_HR.zip` (100 images, ~0.4 GB)
2. Extracts datasets to Google Drive (`/content/drive/MyDrive/SPA-VAEGAN/data/`) and links them to local fast NVMe storage (`/content/data/`).
3. **Persists Patch Banks:** Generates memory-mapped uint8 arrays and copies them to **Google Drive (`/content/drive/MyDrive/SPA-VAEGAN/patches/`)** so you never need to rebuild them on future sessions!


In [ ]:
import os
import urllib.request
import zipfile
import shutil

LOCAL_DATA = "/content/data" if IS_COLAB else "data"
DRIVE_DIR = "/content/drive/MyDrive/SPA-VAEGAN" if IS_COLAB else "."
DRIVE_ZIPS = os.path.join(DRIVE_DIR, "dataset/zips")
DRIVE_DATA = os.path.join(DRIVE_DIR, "data")

os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(DRIVE_ZIPS, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)

DIV2K_URLS = {
    "DIV2K_train_HR.zip": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip",
    "DIV2K_valid_HR.zip": "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip"
}

for zip_name, url in DIV2K_URLS.items():
    folder_name = zip_name.replace(".zip", "")
    drive_folder = os.path.join(DRIVE_DATA, folder_name)
    local_folder = os.path.join(LOCAL_DATA, folder_name)
    drive_zip_path = os.path.join(DRIVE_ZIPS, zip_name)

    # Check if unzipped folder already exists on Google Drive
    if os.path.exists(drive_folder) and len(os.listdir(drive_folder)) > 0:
        print(f"[✓] Found unzipped dataset on Google Drive: {drive_folder}")
        if not os.path.exists(local_folder):
            try:
                os.symlink(drive_folder, local_folder)
                print(f"[✓] Linked {folder_name} to local NVMe disk.")
            except Exception:
                shutil.copytree(drive_folder, local_folder)
                print(f"[✓] Copied {folder_name} to local NVMe disk.")
        continue

    # Download zip to Google Drive if not present
    if not os.path.exists(drive_zip_path):
        print(f"[↓] Downloading {zip_name} to Google Drive ({drive_zip_path})...")
        try:
            urllib.request.urlretrieve(url, drive_zip_path)
            print(f"[✓] Downloaded {zip_name}")
        except Exception as e:
            print(f"[!] Download error: {e}")

    # Extract to Google Drive (persistent) and local disk (fast NVMe)
    if os.path.exists(drive_zip_path):
        print(f"[📦] Extracting {zip_name} to Google Drive ({DRIVE_DATA})...")
        with zipfile.ZipFile(drive_zip_path, 'r') as zf:
            zf.extractall(DRIVE_DATA)
        print(f"[✓] Extracted to Google Drive: {drive_folder}")
        
        if not os.path.exists(local_folder):
            try:
                os.symlink(drive_folder, local_folder)
            except Exception:
                with zipfile.ZipFile(drive_zip_path, 'r') as zf:
                    zf.extractall(LOCAL_DATA)
            print(f"[✓] Extracted to local disk: {local_folder}")


### **Build & Persist Patch Banks to Google Drive**

In [ ]:
LOCAL_PATCHES = "/content/patches" if IS_COLAB else "dataset/patches"
DRIVE_PATCHES = "/content/drive/MyDrive/SPA-VAEGAN/patches" if IS_COLAB else "dataset/patches"

os.makedirs(LOCAL_PATCHES, exist_ok=True)
os.makedirs(DRIVE_PATCHES, exist_ok=True)

# Check if patches already exist in Drive
drive_has_all = all(os.path.exists(os.path.join(DRIVE_PATCHES, f)) 
                    for f in ["train_patches.npy", "val_patches.npy", "test_patches.npy"])

if drive_has_all:
    print("[✓] All patch banks already exist in Google Drive! Copying to local NVMe disk...")
    for f in ["train_patches.npy", "val_patches.npy", "test_patches.npy"]:
        src_f = os.path.join(DRIVE_PATCHES, f)
        dst_f = os.path.join(LOCAL_PATCHES, f)
        if not os.path.exists(dst_f):
            print(f"[↻] Copying {f}...")
            shutil.copy2(src_f, dst_f)
    print("[✓] Patch banks successfully loaded onto local disk.")
else:
    print("Building patch banks from scratch and persisting to Google Drive...")
    !python -m src.make_patches --config configs/default.yaml --patches_dir {LOCAL_PATCHES} --drive_patches_dir {DRIVE_PATCHES}

print("\n--- Verified Patch Banks on Google Drive ---")
for fname in ["train_patches.npy", "val_patches.npy", "test_patches.npy"]:
    p = os.path.join(DRIVE_PATCHES, fname)
    if os.path.exists(p):
        mb = os.path.getsize(p) / (1024 * 1024)
        print(f"[✓ Drive] {fname:20s}: {mb:8.1f} MB (Persisted)")
    else:
        print(f"[!] Missing: {fname}")


### **Visual Degradation Pipeline Test**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from src.degradation import degrade

PATCHES_DIR = "/content/patches" if IS_COLAB else "dataset/patches"
test_npy = os.path.join(PATCHES_DIR, "test_patches.npy")
if not os.path.exists(test_npy) and IS_COLAB:
    test_npy = "/content/drive/MyDrive/SPA-VAEGAN/patches/test_patches.npy"

if os.path.exists(test_npy):
    mmap = np.lib.format.open_memmap(test_npy, mode="r")
    sample_hr = mmap[0] # (3, 256, 256) uint8
    
    hr_t = torch.from_numpy(sample_hr).float().div(255.0).unsqueeze(0)
    lr_t = degrade(hr_t) # (1, 3, 64, 64)
    lr_up = F.interpolate(lr_t, scale_factor=4, mode="bicubic", align_corners=False).clamp(0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(sample_hr.transpose(1, 2, 0))
    axes[0].set_title("Ground Truth HR (256x256)", fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(lr_t[0].permute(1, 2, 0).cpu().numpy())
    axes[1].set_title("Degraded LR Input (64x64)", fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(lr_up[0].permute(1, 2, 0).cpu().numpy())
    axes[2].set_title("Bicubic Upsampling 4x", fontweight="bold")
    axes[2].axis("off")

    os.makedirs("results/figures", exist_ok=True)
    plt.savefig("results/figures/degradation_check.png", dpi=200)
    plt.show()
    print("[✓] Saved results/figures/degradation_check.png")


### **Run Architectural Unit Tests & Overfit Sanity Test**

In [ ]:
# Run unit tests
!python tests/test_shapes.py


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from models.cvae import CVAE
from src.degradation import degrade

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CVAE().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

torch.manual_seed(42)
hr_batch = torch.rand(4, 3, 256, 256, device=dev)
lr_batch = degrade(hr_batch)

print("Running 300-step overfit test...")
losses = []
for step in range(1, 301):
    opt.zero_grad()
    sr, mu, logvar = model(lr_batch, hr_batch)
    loss = F.l1_loss(sr, hr_batch)
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 4))
plt.plot(losses, color="#e74c3c", lw=2, label="L1 Loss")
plt.title("Single-Batch Overfit Dynamics (300 Steps)", fontweight="bold")
plt.xlabel("Iteration")
plt.ylabel("L1 Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

assert losses[-1] < losses[0] * 0.25, "Overfit test failed!"
print(f"[✓] Overfit test passed! Loss collapsed from {losses[0]:.4f} to {losses[-1]:.4f}.")


<a name="phase-2"></a>
# 📉 Phase 2: Baseline Conditional VAE Training (30,000 Iterations)

### **Executive Summary & Scope**
Trains the **Baseline Conditional Variational Autoencoder (CVAE)**:
- **Optimization Budget:** 30,000 generator iterations ($B=16$, $256\times 256$ HR crops).
- **Milestone Checkpoint:** At iteration 10,000, `baseline_it10000.pt` is saved to Google Drive. This milestone is frozen and serves as the **shared warm-start** for SPA-VAEGAN.
- **Auto-Resumption:** If Colab times out, re-running the training cell automatically detects `last.pt` on Google Drive and resumes without restarting!


In [ ]:
# Ensure patches are cached on local fast NVMe disk
LOCAL_PATCHES = "/content/patches" if IS_COLAB else "dataset/patches"
DRIVE_PATCHES = "/content/drive/MyDrive/SPA-VAEGAN/patches" if IS_COLAB else "dataset/patches"
os.makedirs(LOCAL_PATCHES, exist_ok=True)

for fname in ["train_patches.npy", "val_patches.npy"]:
    local_p = os.path.join(LOCAL_PATCHES, fname)
    drive_p = os.path.join(DRIVE_PATCHES, fname)
    if not os.path.exists(local_p) and os.path.exists(drive_p):
        print(f"[↻] Syncing {fname} from Google Drive to local disk...")
        shutil.copy2(drive_p, local_p)
print("[✓] Training patches verified on local disk.")


### **Launch Baseline CVAE Training**

In [ ]:
# Launch Baseline CVAE training
!python -m src.train_baseline --config configs/default.yaml --patches_dir /content/patches


### **Verify Milestone Checkpoint & Plot Baseline Trajectories**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Verify 10k milestone on Drive
milestone_path = "/content/drive/MyDrive/SPA-VAEGAN/checkpoints/baseline/baseline_it10000.pt" if IS_COLAB else "checkpoints/baseline/baseline_it10000.pt"
if os.path.exists(milestone_path):
    print(f"[✓] Milestone checkpoint verified: {milestone_path} ({os.path.getsize(milestone_path)/(1024*1024):.2f} MB)")
else:
    print(f"[!] Warning: Milestone not found at {milestone_path}.")

# Plot training curves
log_csv = "results/logs/baseline.csv"
if os.path.exists(log_csv) and os.path.getsize(log_csv) > 0:
    df = pd.read_csv(log_csv)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(df["iteration"], df["L_pix"], color="#1f77b4", lw=1.5)
    axes[0].set_title("L1 Pixel Loss vs Iterations", fontweight="bold")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("L1 Loss")
    axes[0].grid(True, linestyle="--", alpha=0.6)

    ax_kl = axes[1]
    ax_kl.plot(df["iteration"], df["L_KL"], color="#2ca02c", lw=1.5, label="KL Divergence")
    ax_kl.set_title("KL Divergence & Beta Warmup", fontweight="bold")
    ax_kl.set_xlabel("Iteration")
    ax_kl.set_ylabel("KL Loss", color="#2ca02c")
    ax_kl.grid(True, linestyle="--", alpha=0.6)

    ax_beta = ax_kl.twinx()
    ax_beta.plot(df["iteration"], df["beta_t"], color="#ff7f0e", linestyle="--", label="Beta_t")
    ax_beta.set_ylabel("Beta Weight", color="#ff7f0e")

    val_rows = df[df["val_psnr"].notnull() & (df["val_psnr"] != "")]
    if not val_rows.empty:
        axes[2].plot(val_rows["iteration"], pd.to_numeric(val_rows["val_psnr"]), "o-", color="#d62728", lw=2)
        axes[2].set_title("Validation PSNR (dB)", fontweight="bold")
        axes[2].set_xlabel("Iteration")
        axes[2].set_ylabel("PSNR (dB)")
        axes[2].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.savefig("results/figures/baseline_curves.png", dpi=200)
    plt.show()


<a name="phase-3"></a>
# 🌟 Phase 3: Proposed SPA-VAEGAN Adversarial Training (20,000 Iterations)

### **Executive Summary & Scope**
Trains **SPA-VAEGAN (Structure-Preserving Adaptive VAE-GAN)** to solve the blurriness bottleneck:
1. **Shared Warm-Start Protocol:** The generator is initialised from `baseline_it10000.pt` on Google Drive and trained for 20,000 additional iterations. Both models end with exactly 30,000 generator updates on identical data.
2. **Conditional PatchGAN Discriminator:** Evaluates local $32\times 32$ patch realism with Spectral Normalization and Hinge Loss.
3. **Sobel Edge Structure Loss ($\lambda_e = 0.5$):** Enforces high-frequency gradient fidelity across RGB channels.
4. **Gradient-Ratio Adaptive Adversarial Weighting ($\lambda_t$):** Dynamically scales adversarial gradients to maintain a stable ratio ($\delta = 0.1$) relative to reconstruction gradients.


### **Launch SPA-VAEGAN Adversarial Training**

In [ ]:
# Launch SPA-VAEGAN training
!python -m src.train_spa_vaegan --config configs/default.yaml --patches_dir /content/patches


### **Plot SPA-VAEGAN Stability Diagnostics & Visual Comparison**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

spa_log = "results/logs/spa_vaegan.csv"
if os.path.exists(spa_log) and os.path.getsize(spa_log) > 0:
    df = pd.read_csv(spa_log)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # 1. Losses
    ax1 = axes[0, 0]
    ax1.plot(df["iteration"], df["L_pix"], label="L1 Pixel", color="#1f77b4")
    ax1.plot(df["iteration"], df["L_edge"], label="Sobel Edge", color="#2ca02c")
    ax1.set_title("Reconstruction Losses", fontweight="bold")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()

    # 2. Discriminator
    ax2 = axes[0, 1]
    ax2.plot(df["iteration"], df["L_D"], label="Discriminator Loss (L_D)", color="#d62728")
    margin = pd.to_numeric(df["D_real"]) - pd.to_numeric(df["D_fake"])
    ax2.plot(df["iteration"], margin, label="D(real) - D(fake) Margin", color="#ff7f0e", linestyle="--")
    ax2.set_title("Discriminator Dynamics & Margin", fontweight="bold")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Loss / Margin")
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()

    # 3. Adaptive Weight
    ax3 = axes[1, 0]
    ax3.plot(df["iteration"], df["lambda_t"], label="Adaptive Weight (lambda_t)", color="#9467bd", lw=2)
    ax3.set_title("Adaptive Adversarial Weight (lambda_t)", fontweight="bold")
    ax3.set_xlabel("Iteration")
    ax3.set_ylabel("Lambda")
    ax3.grid(True, linestyle="--", alpha=0.6)
    ax3.legend()

    # 4. Gradient Norms
    ax4 = axes[1, 1]
    ax4.plot(df["iteration"], df["norm_rec"], label="||grad L_rec||", color="#8c564b", alpha=0.7)
    ax4.plot(df["iteration"], df["norm_adv"], label="||grad L_adv||", color="#e377c2", alpha=0.7)
    ax4.set_title("Gradient Norm Dynamics at Reference Layer", fontweight="bold")
    ax4.set_xlabel("Iteration")
    ax4.set_ylabel("L2 Norm")
    ax4.grid(True, linestyle="--", alpha=0.6)
    ax4.legend()

    plt.tight_layout()
    plt.savefig("results/figures/spa_vaegan_curves.png", dpi=200)
    plt.show()


<a name="phase-4"></a>
# 📊 Phase 4: Quantitative Benchmarking, Figures & Deployment

### **Executive Summary & Scope**
1. **Held-Out Test Set Evaluation:** Evaluates Bicubic, Baseline CVAE, and SPA-VAEGAN across all **2,000 held-out test patches**.
2. **Seven Quantitative Metrics:** Computes MSE, MAE, PSNR, SSIM, LPIPS, FID, and Inference Latency.
3. **Statistical Significance Testing:** Computes 95% paired bootstrap confidence intervals (1,000 resamples).
4. **Publication Figures:** Generates multi-panel comparison grids with zoomed crop insets and Sobel edge maps.
5. **Model Weight Export:** Strips optimizer states and discriminators, producing compact ~7 MB generator checkpoints ready for Streamlit deployment.


In [ ]:
# Ensure test patches are on local disk
LOCAL_PATCHES = "/content/patches" if IS_COLAB else "dataset/patches"
DRIVE_PATCHES = "/content/drive/MyDrive/SPA-VAEGAN/patches" if IS_COLAB else "dataset/patches"

local_test = os.path.join(LOCAL_PATCHES, "test_patches.npy")
drive_test = os.path.join(DRIVE_PATCHES, "test_patches.npy")

if not os.path.exists(local_test) and os.path.exists(drive_test):
    print("[↻] Syncing test_patches.npy from Google Drive...")
    shutil.copy2(drive_test, local_test)
print("[✓] Test patches verified on local disk.")


### **Run Comprehensive Test Benchmark Across 2,000 Held-Out Patches**

In [ ]:
# Run 2,000-patch test evaluation
test_npy = "/content/patches/test_patches.npy" if IS_COLAB else "dataset/patches/test_patches.npy"

!python -m evaluation.evaluate --config configs/default.yaml --test_npy {test_npy}


### **Display Final Benchmark Table**

In [ ]:
import pandas as pd

bench_csv = "results/benchmark.csv"
if os.path.exists(bench_csv):
    df = pd.read_csv(bench_csv)
    display(df)
else:
    print("[!] Benchmark CSV not found.")


### **Profile Latency & Generate Publication Figures**

In [ ]:
# Profile latency & parameters
!python -m evaluation.latency --config configs/default.yaml

# Generate visual comparison figures & edge maps
!python -m evaluation.make_figures --test_npy {test_npy} --out_dir results/figures

from IPython.display import Image, display

if os.path.exists("results/figures/comparison_grid.png"):
    print("Visual Comparison Grid:")
    display(Image("results/figures/comparison_grid.png", width=900))

if os.path.exists("results/figures/edge_maps.png"):
    print("\nSobel Edge Map Comparison:")
    display(Image("results/figures/edge_maps.png", width=900))


### **Export Lightweight Generator Models (~7 MB) for Streamlit Deployment**

In [ ]:
# Export Baseline Generator
!python -m src.export_model --input checkpoints/baseline/best.pt --output checkpoints/cvae_baseline_G.pt

# Export SPA-VAEGAN Generator
!python -m src.export_model --input checkpoints/spa_vaegan/best.pt --output checkpoints/spa_vaegan_G.pt

print("\n--- Exported Checkpoint Sizes ---")
for fname in ["cvae_baseline_G.pt", "spa_vaegan_G.pt"]:
    p = os.path.join("checkpoints", fname)
    if os.path.exists(p):
        mb = os.path.getsize(p) / (1024 * 1024)
        print(f"{fname:22s}: {mb:.2f} MB")


### **Streamlit Web Application Smoke Test**

In [ ]:
import numpy as np
from PIL import Image
from app.inference import load_model, upscale_tiled

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("checkpoints/spa_vaegan_G.pt", device=str(dev))

# Smoke test on synthetic sample image
sample_img = Image.open("app/assets/sample.png").convert("RGB")
img_np = np.array(sample_img)

sr_np, latency_ms = upscale_tiled(model, img_np, device=dev, tile=64, halo=24)
print(f"[✓] Tiled inference successful! Output shape: {sr_np.shape}, Elapsed: {latency_ms:.1f} ms")

sr_img = Image.fromarray(sr_np)
display(sr_img)


### 🔍 **Pipeline Complete!**
To run the interactive Streamlit application locally on your PC:
```bash
streamlit run app/app.py
```
